In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

from layout_ancoras import gerar_ancoras

# output/<localizacao>/<caso>/saida.csv — gerado por
# scripts/rodar_experimentos_ruido.sh. O script grava em
# <raiz do pacote>/output/ (não scripts/output/), então o caminho é
# relativo à raiz do pacote, um nível acima deste notebook.
RESULTADOS_DIR = Path("../output")

# Mesmo mapeamento caso -> n_ancoras da tabela "Casos de ruído para
# avaliação" do README / scripts/rodar_experimentos_ruido.sh — precisa
# pra saber onde as âncoras daquele caso ficaram (varia por caso).
N_ANCORAS_POR_CASO = {"default": 10, "simples": 6, "medio": 8, "elevado": 10}
ORDEM_CASOS = ["default", "simples", "medio", "elevado"]

execucoes = sorted(RESULTADOS_DIR.glob("*/*/saida.csv"))
if not execucoes:
    raise FileNotFoundError(
        f"Nenhum saida.csv encontrado em {RESULTADOS_DIR}/<localizacao>/<caso>/ "
        "— rode scripts/rodar_experimentos_ruido.sh primeiro (ou ajuste "
        "RESULTADOS_DIR pra apontar pra outra pasta)."
    )

print(f"{len(execucoes)} execuções encontradas em {RESULTADOS_DIR}/:")
for caminho in execucoes:
    print(" ", caminho)


In [ ]:
from dataclasses import dataclass
from typing import Dict

@dataclass(frozen=True)
class Anchor:
    x: float
    y: float


def anchors_do_caso(caso: str) -> Dict[int, Anchor]:
    """Posições das âncoras daquele caso, geradas por
    layout_ancoras.gerar_ancoras() (mesma fonte que scripts/spawn_campo.py
    usa pra spawnar no Gazebo) — cada caso do experimento roda com um
    n_ancoras diferente (ver N_ANCORAS_POR_CASO), então as posições não
    são as mesmas entre casos."""
    n = N_ANCORAS_POR_CASO.get(caso, 10)
    return {a["id"]: Anchor(a["x"], a["y"]) for a in gerar_ancoras(n)}


In [ ]:
def carregar_execucao(caminho_csv: Path) -> pd.DataFrame:
    """Lê um saida.csv de scripts/bag_para_csv.py (caminho
    output/<localizacao>/<caso>/saida.csv) e deriva as mesmas colunas de
    antes: pose estimada/real numéricas e, por linha, o checkpoint
    (âncora de menor range lida naquele instante, entre as que
    reportaram — não existe coluna única de checkpoint no CSV bruto)."""
    caso = caminho_csv.parent.name
    localizacao = caminho_csv.parent.parent.name
    anchors = anchors_do_caso(caso)

    df = pd.read_csv(caminho_csv)
    df["Robot_X_est"] = pd.to_numeric(df["filtrada_x"], errors="coerce")
    df["Robot_Y_est"] = pd.to_numeric(df["filtrada_y"], errors="coerce")
    df["Robot_X_real"] = pd.to_numeric(df["ground_truth_x"], errors="coerce")
    df["Robot_Y_real"] = pd.to_numeric(df["ground_truth_y"], errors="coerce")

    range_cols = ["anchor%d_range" % i for i in sorted(anchors) if "anchor%d_range" % i in df.columns]
    for c in range_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    def calcula_checkpoint(row):
        valores = {int(c[len("anchor"):-len("_range")]): row[c]
                  for c in range_cols if pd.notna(row[c])}
        if not valores:
            return pd.Series({"Checkpoint": np.nan, "Checkpoint_Dist_Sensor": np.nan})
        cid = min(valores, key=valores.get)
        return pd.Series({"Checkpoint": cid, "Checkpoint_Dist_Sensor": valores[cid]})

    df[["Checkpoint", "Checkpoint_Dist_Sensor"]] = df.apply(calcula_checkpoint, axis=1)
    df["Checkpoint_X"] = df["Checkpoint"].map(lambda cid: anchors[int(cid)].x if pd.notna(cid) else np.nan)
    df["Checkpoint_Y"] = df["Checkpoint"].map(lambda cid: anchors[int(cid)].y if pd.notna(cid) else np.nan)

    # Distância real = geometria entre a pose real (ground truth) e a
    # posição conhecida da âncora — não vem do sensor, serve pra comparar
    # com Checkpoint_Dist_Sensor (ambas em metros).
    df["Checkpoint_Dist_Real"] = np.sqrt(
        (df["Robot_X_real"] - df["Checkpoint_X"])**2 + (df["Robot_Y_real"] - df["Checkpoint_Y"])**2
    )

    df["localizacao"] = localizacao
    df["caso"] = caso
    return df


# {(localizacao, caso): DataFrame} — uma entrada por execução encontrada
execucoes_df = {
    (caminho.parent.parent.name, caminho.parent.name): carregar_execucao(caminho)
    for caminho in execucoes
}

localizacoes = sorted({loc for loc, _ in execucoes_df})
casos_presentes = [c for c in ORDEM_CASOS if any(c == caso for _, caso in execucoes_df)]


In [ ]:
def plot_trajetoria(df: pd.DataFrame, anchors: Dict[int, Anchor], titulo: str, ax) -> None:
    """Trajetória estimada x real, âncoras e um círculo por checkpoint
    (raio = metade da distância entre o 1o e o último ponto em que aquele
    id foi o mais próximo — mesma ideia de antes, agora reutilizável por
    execução)."""
    traj = df.dropna(subset=["Robot_X_est", "Robot_Y_est", "Robot_X_real", "Robot_Y_real"])
    ax.plot(traj["Robot_X_est"].to_numpy(), traj["Robot_Y_est"].to_numpy(), linestyle="-", linewidth=0.3,
            marker="o", markersize=1.2, color="blue", label="Estimated")
    ax.plot(traj["Robot_X_real"].to_numpy(), traj["Robot_Y_real"].to_numpy(), linestyle="-", linewidth=0.3,
            marker="o", markersize=1.2, color="red", label="Real")

    cmap = plt.get_cmap("tab10")
    colors = {cid: cmap(i % 10) for i, cid in enumerate(anchors.keys())}
    for cid, a in anchors.items():
        ax.scatter(a.x, a.y, marker="x", s=55, color=colors[cid])
        ax.annotate(str(cid), (a.x, a.y), textcoords="offset points", xytext=(4, 4), fontsize=7)

    # Círculo por checkpoint (raio = metade da distância entre o 1o e o
    # último ponto em que aquele id foi o mais próximo) — comentado por
    # enquanto, poluía o gráfico de trajetória.
    # valid = df.dropna(subset=["Robot_X_est", "Robot_Y_est", "Checkpoint"]).copy()
    # valid["Checkpoint"] = valid["Checkpoint"].astype(int)
    # for cid in sorted(valid["Checkpoint"].unique()):
    #     sub = valid[valid["Checkpoint"] == cid].sort_values("t")
    #     x0, y0 = sub["Robot_X_est"].iloc[0], sub["Robot_Y_est"].iloc[0]
    #     x1, y1 = sub["Robot_X_est"].iloc[-1], sub["Robot_Y_est"].iloc[-1]
    #     centro = ((x0 + x1) / 2.0, (y0 + y1) / 2.0)
    #     raio = max(math.hypot(x1 - x0, y1 - y0) / 2.0, 0.01)
    #     ax.add_patch(plt.Circle(centro, raio, fill=False, linewidth=1.2, color=colors.get(cid, cmap(0))))

    ax.set_title(titulo, fontsize=11)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.axis("equal")
    ax.xaxis.set_major_locator(MultipleLocator(0.2))
    ax.yaxis.set_major_locator(MultipleLocator(0.2))
    ax.grid(True, linewidth=0.5, alpha=0.6)
    ax.autoscale_view()


# Uma figura por vez (não mais uma grade única com tudo) — mais fácil de
# olhar e de exportar individualmente. Ordem: localizacao, depois caso.
for loc in localizacoes:
    for caso in casos_presentes:
        df_exec = execucoes_df.get((loc, caso))
        if df_exec is None:
            continue
        fig, ax = plt.subplots(figsize=(6, 6))
        plot_trajetoria(df_exec, anchors_do_caso(caso), f"{loc} — {caso}", ax)
        ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
        plt.tight_layout()
        plt.show()


In [ ]:
def resume_execucao(df: pd.DataFrame) -> dict:
    """Uma linha de métricas (RMSE de posição/distância, amostras,
    duração) pra uma execução — mesmo cálculo de antes, agora por
    (localizacao, caso) em vez de um único df global."""
    pos = df.dropna(subset=["Robot_X_est", "Robot_Y_est", "Robot_X_real", "Robot_Y_real"])
    dx = pos["Robot_X_est"].to_numpy() - pos["Robot_X_real"].to_numpy()
    dy = pos["Robot_Y_est"].to_numpy() - pos["Robot_Y_real"].to_numpy()

    dist = df.dropna(subset=["Checkpoint_Dist_Sensor", "Checkpoint_Dist_Real"])
    dd = (dist["Checkpoint_Dist_Sensor"].to_numpy() - dist["Checkpoint_Dist_Real"].to_numpy()
          if len(dist) else np.array([]))

    return {
        "localizacao": df["localizacao"].iloc[0],
        "caso": df["caso"].iloc[0],
        "n_amostras": len(pos),
        "RMSE_x_m": float(np.sqrt(np.mean(dx**2))) if len(dx) else np.nan,
        "RMSE_y_m": float(np.sqrt(np.mean(dy**2))) if len(dy) else np.nan,
        "RMSE_pos2d_m": float(np.sqrt(np.mean(dx**2 + dy**2))) if len(dx) else np.nan,
        "RMSE_dist_m": float(np.sqrt(np.mean(dd**2))) if len(dd) else np.nan,
        "duracao_s": float(df["t"].max() - df["t"].min()),
    }


resumo = pd.DataFrame(resume_execucao(df) for df in execucoes_df.values())
resumo["RMSE_pos2d_cm"] = resumo["RMSE_pos2d_m"] * 100.0
resumo["caso"] = pd.Categorical(resumo["caso"], categories=ORDEM_CASOS, ordered=True)
resumo = resumo.sort_values(["caso", "localizacao"]).reset_index(drop=True)

resumo


In [ ]:
def resume_por_checkpoint(df: pd.DataFrame) -> pd.DataFrame:
    """RMSE de posição e de distância por checkpoint dentro de uma única
    execução — mesmo cálculo de antes (groupby por Checkpoint), com
    localizacao/caso anexados pra poder concatenar entre execuções."""
    pos = df.dropna(subset=["Robot_X_est", "Robot_Y_est", "Robot_X_real", "Robot_Y_real"]).copy()
    pos["dx"] = pos["Robot_X_est"] - pos["Robot_X_real"]
    pos["dy"] = pos["Robot_Y_est"] - pos["Robot_Y_real"]
    pos["err_pos2d"] = np.sqrt(pos["dx"]**2 + pos["dy"]**2)

    dist = df.dropna(subset=["Checkpoint_Dist_Sensor", "Checkpoint_Dist_Real"]).copy()
    dist["err_dist"] = dist["Checkpoint_Dist_Sensor"] - dist["Checkpoint_Dist_Real"]

    pos_rmse = pos.groupby("Checkpoint").agg(
        n_pos=("err_pos2d", "size"),
        rmse_x=("dx", lambda s: float(np.sqrt(np.mean(np.square(s.to_numpy()))))),
        rmse_y=("dy", lambda s: float(np.sqrt(np.mean(np.square(s.to_numpy()))))),
        rmse_pos2d=("err_pos2d", lambda s: float(np.sqrt(np.mean(np.square(s.to_numpy()))))),
    )
    dist_rmse = dist.groupby("Checkpoint").agg(
        n_dist=("err_dist", "size"),
        rmse_dist=("err_dist", lambda s: float(np.sqrt(np.mean(np.square(s.to_numpy()))))),
    )

    tabela = pos_rmse.join(dist_rmse, how="outer").reset_index()
    tabela.insert(0, "caso", df["caso"].iloc[0])
    tabela.insert(0, "localizacao", df["localizacao"].iloc[0])
    return tabela


resumo_checkpoints = pd.concat(
    (resume_por_checkpoint(df) for df in execucoes_df.values()),
    ignore_index=True,
)
resumo_checkpoints["caso"] = pd.Categorical(resumo_checkpoints["caso"], categories=ORDEM_CASOS, ordered=True)
resumo_checkpoints = resumo_checkpoints.sort_values(
    ["localizacao", "caso", "rmse_pos2d"], ascending=[True, True, False]
).reset_index(drop=True)

print(resumo_checkpoints.to_string(index=False))


In [ ]:
def grafico_comparativo(coluna: str, titulo: str, ylabel: str) -> None:
    """Barras agrupadas: uma barra por localizacao, um grupo por caso —
    lê direto de `resumo` em vez de números colados na mão."""
    x = np.arange(len(casos_presentes))
    largura = 0.8 / max(len(localizacoes), 1)

    plt.figure()
    for i, loc in enumerate(localizacoes):
        valores = [
            resumo.loc[(resumo["caso"] == caso) & (resumo["localizacao"] == loc), coluna]
            .pipe(lambda s: s.iloc[0] if len(s) else np.nan)
            for caso in casos_presentes
        ]
        plt.bar(x + i * largura, valores, largura, label=loc)

    plt.title(titulo)
    plt.xlabel("Caso de ruído")
    plt.ylabel(ylabel)
    plt.xticks(x + largura * (len(localizacoes) - 1) / 2, casos_presentes)
    plt.legend()
    plt.tight_layout()
    plt.show()


grafico_comparativo("RMSE_pos2d_cm", "RMSE de posição por caso de ruído", "RMSE 2D (cm)")
grafico_comparativo("RMSE_dist_m", "RMSE de distância ao checkpoint por caso de ruído", "RMSE distância (m)")
